In [1]:
#stage3 -fine tune
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader

from model import WDMPNN, GraphPredictor
from data_preparation import PolymerDataset

from data_preparation import load_and_split_data, smiles_to_data

def prepare_property_datasets(properties, base_path="neurips-open-polymer-prediction-2025"):
    """
    加载原始 train/val/test，并为每个属性返回清洗好的 DataFrame。
    返回值示例：
    {
      "Tg": {"train": train_clean_df, "val": val_clean_df, "test": test_clean_df},
      "FFV": { ... },
      ...
    }
    """
    train_df, val_df, test_df = load_and_split_data(base_path)
    result = {}
    for prop in properties:
        train_clean = train_df[["SMILES", prop]].dropna().reset_index(drop=True)
        val_clean   = val_df[  ["SMILES", prop]].dropna().reset_index(drop=True)
        test_clean  = test_df[ ["SMILES", prop]].dropna().reset_index(drop=True)
        result[prop] = {
            "train": train_clean,
            "val":   val_clean,
            "test":  test_clean
        }
    return result
    
def finetune_property(
    train_df,
    val_df,
    property_name: str,
    best_params_path: str = "stage1_best_params.pt",
    stage2_encoder_path: str = "stage2_encoder.pt",
    stage2_predictor_path: str = "stage2_predictor.pt",
    output_dir: str = "stage3_heads",
    device: torch.device = None,
    num_epochs: int = 50,
    batch_size: int = 64,
    patience: int = 10
):
    # 设备选择
    device = device or (torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))

    # 准备数据集
    train_ds = PolymerDataset(train_df, y_cols=[property_name])
    val_ds   = PolymerDataset(val_df,   y_cols=[property_name])
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

    # 加载 stage1 超参 & stage2 模型
    best = torch.load(best_params_path)
    encoder = WDMPNN(2, 1, best["hidden_dim"], best["num_edge_layers"]).to(device)
    encoder.load_state_dict(torch.load(stage2_encoder_path))
    predictor = GraphPredictor(encoder.hidden_dim, best["hidden_dim"]//2, 1).to(device)
    predictor.load_state_dict(torch.load(stage2_predictor_path))

    # 下游 head
    downstream = nn.Sequential(nn.Linear(1,32), nn.ReLU(), nn.Linear(32,1)).to(device)

    # 全量微调
    optim = torch.optim.Adam(
        list(encoder.parameters())+
        list(predictor.parameters())+
        list(downstream.parameters()),
        lr=best.get("lr",1e-3)
    )

    best_val_mae = float('inf')
    no_improve = 0

    for epoch in range(1, num_epochs+1):
        # —— 训练一步 —— 
        encoder.train(); predictor.train(); downstream.train()

        total_abs_error = 0.0
        total_mse_loss = 0.0
        for data in train_loader:
            data = data.to(device)
            h = encoder(data.x, data.edge_index, data.edge_attr,
                        torch.ones(data.edge_attr.size(0),device=device),
                        data.batch)
            h = predictor(h).view(-1,1)
            out = downstream(h)
            y = data.y.view(-1,1)
            loss = F.mse_loss(out, y)
            optim.zero_grad(); loss.backward(); optim.step()
            total_mse_loss   += loss.item() * data.num_graphs
            total_abs_error  += torch.abs(out - y).sum().item()
        train_mse = total_mse_loss / len(train_loader.dataset)
        train_mae = total_abs_error  / len(train_loader.dataset)

        # —— 验证集评估 —— 
        encoder.eval(); predictor.eval(); downstream.eval()
        val_mse_loss = 0.0
        val_abs_error = 0.0
        with torch.no_grad():
            for data in val_loader:
                data = data.to(device)
                h = predictor(encoder(data.x, data.edge_index, data.edge_attr,
                                      torch.ones(data.edge_attr.size(0),device=device),
                                      data.batch)).view(-1,1)
                out = downstream(h)
                y = data.y.view(-1,1)
                val_mse_loss  += F.mse_loss(out, y, reduction='sum').item()
                val_abs_error += torch.abs(out - y).sum().item()

        val_mse = val_mse_loss / len(val_loader.dataset)
        val_mae = val_abs_error / len(val_loader.dataset)

        print(f"[{property_name}] Epoch {epoch:03d} Train MSE={train_mse:.4f}, MAE={train_mae:.4f}, Val   MSE={val_mse:.4f}, MAE={val_mae:.4f}")

        # —— 早停判断 —— 
        if val_mae < best_val_mae - 1e-4:
            best_val_mae = val_mae
            no_improve = 0
            # 保存当前最优 head
            best_state = downstream.state_dict()
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch} (no val improvement in {patience} epochs)")
                break

    # 保存在验证集上最优的 downstream head
    os.makedirs(output_dir, exist_ok=True)

    # 1) 保存最优 downstream head
    torch.save(best_state,
               os.path.join(output_dir, f"downstream_{property_name}.pt"))

    # 2) 保存微调后的 encoder
    torch.save(encoder.state_dict(),
               os.path.join(output_dir, f"encoder_ft_{property_name}.pt"))

    # 3) 保存微调后的 predictor
    torch.save(predictor.state_dict(),
               os.path.join(output_dir, f"predictor_ft_{property_name}.pt"))

    print(f"Saved head & encoder & predictor for {property_name} in {output_dir}")

In [2]:
properties = ["Tg", "FFV", "Tc", "Density", "Rg"]
datasets = prepare_property_datasets(properties)

👉 加载主训练数据
  原始训练样本数: 7973
  → 正在增强 Tc 数据，共 874 条
cross_smiles: 737
    填充已有样本 0 条，新增样本 129 条
  → 正在增强 Tg 数据，共 662 条
cross_smiles: 526
    填充已有样本 15 条，新增样本 136 条
  → 正在增强 Tg 数据，共 501 条
cross_smiles: 0
    填充已有样本 0 条，新增样本 499 条
  → 正在增强 Density 数据，共 787 条
cross_smiles: 254
    填充已有样本 110 条，新增样本 525 条
  → 正在增强 FFV 数据，共 862 条


[17:36:31] SMILES Parse Error: syntax error while parsing: *O[Si](*)([R])[R]
[17:36:31] SMILES Parse Error: Failed parsing SMILES '*O[Si](*)([R])[R]' for input: '*O[Si](*)([R])[R]'
[17:36:31] SMILES Parse Error: syntax error while parsing: *NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4
[17:36:31] SMILES Parse Error: Failed parsing SMILES '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4' for input: '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4'
[17:36:31] SMILES Parse Error: syntax error while parsing: O=C=N[R1]N=C=O.O[R2]O.O[R3]O
[17:36:31] SMILES Parse Error: Failed parsing SMILES 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O' for input: 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O'
[17:36:31] SMILES Parse Error: syntax error while parsing: *CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O
[17:36:31] SMILES Parse Error: Failed parsing SMILES '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O' for input: '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O'
[17:36:31] SMILES Parse 

cross_smiles: 43
    填充已有样本 43 条，新增样本 819 条
add dataset4: 10081
👉 划分 train / validation / test
  划分结果: train=8064, val=1008, test=1009


In [3]:
tg_data = datasets["Tg"]
train_tg, val_tg, test_tg = tg_data["train"], tg_data["val"], tg_data["test"]

In [4]:
finetune_property(
    train_tg,
    val_tg,
    property_name="Tg",
    best_params_path="stage1_best_params.pt",
    stage2_encoder_path="stage2_encoder.pt",
    stage2_predictor_path="stage2_predictor.pt",
    output_dir="stage3_heads",
    num_epochs=150,
    batch_size=64,
    patience=15
)

📦 构建 PolymerDataset，样本数=924
   成功转换为图数据: 924 条
📦 构建 PolymerDataset，样本数=111
   成功转换为图数据: 111 条
[Tg] Epoch 001 Train MSE=16414.8229, MAE=96.1714, Val   MSE=10466.1391, MAE=78.7198
[Tg] Epoch 002 Train MSE=12569.3808, MAE=87.9073, Val   MSE=9952.7477, MAE=73.1138
[Tg] Epoch 003 Train MSE=12115.0850, MAE=83.7595, Val   MSE=10399.8164, MAE=79.0914
[Tg] Epoch 004 Train MSE=11524.2451, MAE=82.1180, Val   MSE=10319.6700, MAE=78.9356
[Tg] Epoch 005 Train MSE=10830.7760, MAE=81.6286, Val   MSE=8899.0915, MAE=70.3744
[Tg] Epoch 006 Train MSE=10099.7689, MAE=78.2311, Val   MSE=9201.0239, MAE=71.7598
[Tg] Epoch 007 Train MSE=9579.1363, MAE=76.9294, Val   MSE=8524.5977, MAE=70.4998
[Tg] Epoch 008 Train MSE=8917.0955, MAE=73.9145, Val   MSE=9253.5144, MAE=71.3022
[Tg] Epoch 009 Train MSE=8609.5011, MAE=72.7767, Val   MSE=7540.9758, MAE=66.3004
[Tg] Epoch 010 Train MSE=8549.6561, MAE=73.1547, Val   MSE=7550.1132, MAE=64.8377
[Tg] Epoch 011 Train MSE=8260.6942, MAE=71.7112, Val   MSE=7102.4490, MAE=64.

In [5]:
ffv_data = datasets["FFV"]
train_ffv, val_ffv, test_ffv = ffv_data["train"], ffv_data["val"], ffv_data["test"]

In [6]:
finetune_property(
    train_ffv,
    val_ffv,
    property_name="FFV",
    best_params_path="stage1_best_params.pt",
    stage2_encoder_path="stage2_encoder.pt",
    stage2_predictor_path="stage2_predictor.pt",
    output_dir="stage3_heads",
    num_epochs=150,
    batch_size=64,
    patience=15
)

📦 构建 PolymerDataset，样本数=6320
   成功转换为图数据: 6320 条
📦 构建 PolymerDataset，样本数=794
   成功转换为图数据: 794 条
[FFV] Epoch 001 Train MSE=18.8480, MAE=1.2401, Val   MSE=0.0016, MAE=0.0329
[FFV] Epoch 002 Train MSE=0.0011, MAE=0.0244, Val   MSE=0.0007, MAE=0.0207
[FFV] Epoch 003 Train MSE=0.0008, MAE=0.0201, Val   MSE=0.0007, MAE=0.0190
[FFV] Epoch 004 Train MSE=0.0008, MAE=0.0202, Val   MSE=0.0007, MAE=0.0205
[FFV] Epoch 005 Train MSE=0.0008, MAE=0.0201, Val   MSE=0.0007, MAE=0.0196
[FFV] Epoch 006 Train MSE=0.0008, MAE=0.0200, Val   MSE=0.0007, MAE=0.0197
[FFV] Epoch 007 Train MSE=0.0008, MAE=0.0202, Val   MSE=0.0007, MAE=0.0192
[FFV] Epoch 008 Train MSE=0.0008, MAE=0.0200, Val   MSE=0.0007, MAE=0.0197
[FFV] Epoch 009 Train MSE=0.0008, MAE=0.0199, Val   MSE=0.0007, MAE=0.0204
[FFV] Epoch 010 Train MSE=0.0008, MAE=0.0198, Val   MSE=0.0007, MAE=0.0190
[FFV] Epoch 011 Train MSE=0.0008, MAE=0.0201, Val   MSE=0.0007, MAE=0.0188
[FFV] Epoch 012 Train MSE=0.0009, MAE=0.0206, Val   MSE=0.0010, MAE=0.0234
[FF

In [7]:
Tc_data = datasets["Tc"]
train_Tc, val_Tc, test_Tc = Tc_data["train"], Tc_data["val"], Tc_data["test"]

In [8]:
finetune_property(
    train_Tc,
    val_Tc,
    property_name="Tc",
    best_params_path="stage1_best_params.pt",
    stage2_encoder_path="stage2_encoder.pt",
    stage2_predictor_path="stage2_predictor.pt",
    output_dir="stage3_heads",
    num_epochs=150,
    batch_size=64,
    patience=15
)

📦 构建 PolymerDataset，样本数=707
   成功转换为图数据: 707 条
📦 构建 PolymerDataset，样本数=81
   成功转换为图数据: 81 条
[Tc] Epoch 001 Train MSE=829.3606, MAE=17.9105, Val   MSE=0.0175, MAE=0.1064
[Tc] Epoch 002 Train MSE=0.3713, MAE=0.4810, Val   MSE=0.2394, MAE=0.3693
[Tc] Epoch 003 Train MSE=0.1150, MAE=0.2222, Val   MSE=0.0253, MAE=0.1065
[Tc] Epoch 004 Train MSE=0.0271, MAE=0.0964, Val   MSE=0.0126, MAE=0.0822
[Tc] Epoch 005 Train MSE=0.0155, MAE=0.0861, Val   MSE=0.0114, MAE=0.0818
[Tc] Epoch 006 Train MSE=0.0132, MAE=0.0887, Val   MSE=0.0124, MAE=0.0813
[Tc] Epoch 007 Train MSE=0.0131, MAE=0.0850, Val   MSE=0.0129, MAE=0.0896
[Tc] Epoch 008 Train MSE=0.0130, MAE=0.0905, Val   MSE=0.0101, MAE=0.0732
[Tc] Epoch 009 Train MSE=0.0105, MAE=0.0817, Val   MSE=0.0167, MAE=0.0946
[Tc] Epoch 010 Train MSE=0.0105, MAE=0.0818, Val   MSE=0.0122, MAE=0.0782
[Tc] Epoch 011 Train MSE=0.0101, MAE=0.0794, Val   MSE=0.0123, MAE=0.0779
[Tc] Epoch 012 Train MSE=0.0089, MAE=0.0771, Val   MSE=0.0123, MAE=0.0781
[Tc] Epoch 013 Tr

In [9]:
Density_data = datasets["Density"]
train_Density, val_Density, test_Density = Density_data["train"], Density_data["val"], Density_data["test"]

In [10]:
finetune_property(
    train_Density,
    val_Density,
    property_name="Density",
    best_params_path="stage1_best_params.pt",
    stage2_encoder_path="stage2_encoder.pt",
    stage2_predictor_path="stage2_predictor.pt",
    output_dir="stage3_heads",
    num_epochs=150,
    batch_size=64,
    patience=15
)

📦 构建 PolymerDataset，样本数=1025
   成功转换为图数据: 1025 条
📦 构建 PolymerDataset，样本数=112
   成功转换为图数据: 112 条
[Density] Epoch 001 Train MSE=5.1074, MAE=1.4842, Val   MSE=0.7897, MAE=0.8635
[Density] Epoch 002 Train MSE=0.5558, MAE=0.6600, Val   MSE=0.3198, MAE=0.4922
[Density] Epoch 003 Train MSE=0.2477, MAE=0.4177, Val   MSE=0.1994, MAE=0.3729
[Density] Epoch 004 Train MSE=0.1499, MAE=0.3095, Val   MSE=0.1119, MAE=0.2668
[Density] Epoch 005 Train MSE=0.0768, MAE=0.2101, Val   MSE=0.0434, MAE=0.1694
[Density] Epoch 006 Train MSE=0.0498, MAE=0.1802, Val   MSE=0.0385, MAE=0.1601
[Density] Epoch 007 Train MSE=0.0450, MAE=0.1630, Val   MSE=0.0388, MAE=0.1600
[Density] Epoch 008 Train MSE=0.0421, MAE=0.1594, Val   MSE=0.0344, MAE=0.1516
[Density] Epoch 009 Train MSE=0.0404, MAE=0.1567, Val   MSE=0.0325, MAE=0.1483
[Density] Epoch 010 Train MSE=0.0372, MAE=0.1510, Val   MSE=0.0283, MAE=0.1406
[Density] Epoch 011 Train MSE=0.0304, MAE=0.1346, Val   MSE=0.0196, MAE=0.1151
[Density] Epoch 012 Train MSE=0.024

In [11]:
Rg_data = datasets["Rg"]
train_Rg, val_Rg, test_Rg = Rg_data["train"], Rg_data["val"], Rg_data["test"]

In [12]:
finetune_property(
    train_Rg,
    val_Rg,
    property_name="Rg",
    best_params_path="stage1_best_params.pt",
    stage2_encoder_path="stage2_encoder.pt",
    stage2_predictor_path="stage2_predictor.pt",
    output_dir="stage3_heads",
    num_epochs=150,
    batch_size=64,
    patience=15
)

📦 构建 PolymerDataset，样本数=519
   成功转换为图数据: 519 条
📦 构建 PolymerDataset，样本数=49
   成功转换为图数据: 49 条
[Rg] Epoch 001 Train MSE=81.1551, MAE=7.0024, Val   MSE=97.4026, MAE=8.4850
[Rg] Epoch 002 Train MSE=86.1686, MAE=7.5183, Val   MSE=97.0471, MAE=6.8203
[Rg] Epoch 003 Train MSE=66.7006, MAE=6.3340, Val   MSE=73.9915, MAE=7.0356
[Rg] Epoch 004 Train MSE=62.1315, MAE=6.1002, Val   MSE=64.9596, MAE=6.0162
[Rg] Epoch 005 Train MSE=55.2702, MAE=5.6713, Val   MSE=57.9992, MAE=5.7985
[Rg] Epoch 006 Train MSE=49.9591, MAE=5.3537, Val   MSE=52.6763, MAE=5.3389
[Rg] Epoch 007 Train MSE=44.5101, MAE=5.0217, Val   MSE=42.9761, MAE=4.9165
[Rg] Epoch 008 Train MSE=41.3026, MAE=4.7161, Val   MSE=41.9325, MAE=4.9517
[Rg] Epoch 009 Train MSE=38.2232, MAE=4.4470, Val   MSE=36.2577, MAE=4.5870
[Rg] Epoch 010 Train MSE=31.1676, MAE=4.0631, Val   MSE=26.2646, MAE=4.0346
[Rg] Epoch 011 Train MSE=27.1996, MAE=3.8611, Val   MSE=24.8140, MAE=4.1007
[Rg] Epoch 012 Train MSE=25.4766, MAE=3.7576, Val   MSE=23.9440, MAE=3.9

In [21]:
from predict_stage3 import predict_for_smiles
import pandas as pd

In [22]:
# 五个属性
PROPERTIES = ["Tg", "FFV", "Tc", "Density", "Rg"]

# 全局超参和设备
BEST_PARAMS = torch.load("stage1_best_params.pt")
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Stage 3 微调后权重目录
STAGE3_DIR  = "stage3_heads"

# 缓存每个属性的三部分模型
_model_cache = {}

In [23]:
# 1) 读取 test.csv
test_csv = os.path.join("neurips-open-polymer-prediction-2025", "test.csv")
df_test  = pd.read_csv(test_csv, dtype={"id": str})

In [24]:
# 2) 对每条 SMILES 预测
out_records = []
for _, row in df_test.iterrows():
    _id, smi = row["id"], row["SMILES"]
    try:
        preds = predict_for_smiles(smi)
    except Exception:
        # 若解析或推理失败，则填 NaN
        preds = {p: float("nan") for p in PROPERTIES}
    rec = {"id": _id}
    rec.update(preds)
    out_records.append(rec)

In [25]:
# 3) 输出 submission.csv
df_out = pd.DataFrame(out_records, columns=["id"] + PROPERTIES)
df_out.to_csv("submission.csv", index=False)
print("Saved submission.csv with", len(df_out), "rows.")

Saved submission.csv with 3 rows.
